# Entrenar Temis en Colab

Entrena una corrida de Temis en una GPU de Colab y guarda el modelo en tu Google Drive. La exportación a ONNX, la calibración y las evaluaciones se hacen después en el computador de casa.

**Antes de empezar**
1. Sube `temis-datos.tar.gz` a la carpeta `temis` de tu Drive (`Mi unidad/temis/temis-datos.tar.gz`). Son los datos que no están en git: los casos procedurales y los dos archivos extra.
2. Elige una GPU con bf16: **Entorno de ejecución → Cambiar tipo de entorno → A100 o L4**. La T4 no sirve (no tiene bf16).
3. Ejecuta las celdas en orden. Deja la pestaña abierta y el portátil sin suspenderse hasta que termine.

El modelo se guarda en `Mi unidad/temis/runs/<corrida>/` al final de cada época que mejora la validación. Si Colab corta, lo guardado hasta ese momento queda en Drive.

In [ ]:
RUN = "v1114-mrbert-general"         # nombre de la corrida (runs/<RUN>)
REF = "main"                          # rama o commit del repositorio
DRIVE = "/content/drive/MyDrive/temis"
BUNDLE_SHA = "5a4d7d39c74a9a95"       # primeros 16 caracteres del sha256 de temis-datos.tar.gz
EXTRA = ["data/user_targets.jsonl", "data/reviewer_targets.jsonl",
         "runs/_data/td_train_extra.jsonl", "runs/_data/general_extra.jsonl"]
OVERWRITE = False                     # True para reentrenar una corrida que ya tiene model.pt en Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución → Cambiar tipo de entorno"
gpu = torch.cuda.get_device_properties(0)
mem_gb = gpu.total_memory / 2**30
print(f"{gpu.name} · {mem_gb:.0f} GB · bf16: {torch.cuda.is_bf16_supported()} · torch {torch.__version__}")
assert torch.cuda.is_bf16_supported(), "Esta GPU no tiene bf16 (T4): elige A100 o L4"
# Con 20 GB o más no hace falta el gradient checkpointing (mismo resultado, más rápido)
CKPT_TOKENS = 10**9 if mem_gb >= 20 else 5120

In [ ]:
%cd /content
!rm -rf temis && git clone -q https://github.com/i-montes/temis && cd temis && git checkout -q {REF} && git log --oneline -1
!pip -q install "transformers==5.22.2" "datasets==5.2.0" "tokenizers==0.23.3" "huggingface_hub==2.4.0"
%cd /content/temis

In [ ]:
import hashlib, os, pathlib, subprocess
bundle = pathlib.Path(DRIVE) / "temis-datos.tar.gz"
assert bundle.exists(), f"Falta {bundle}: súbelo a Mi unidad/temis/"
sha = hashlib.sha256(bundle.read_bytes()).hexdigest()
assert sha.startswith(BUNDLE_SHA), f"temis-datos.tar.gz no es el esperado ({sha[:16]})"
subprocess.run(["tar", "-xzf", str(bundle)], check=True)
for f in ["data/procedural/cases.jsonl"] + EXTRA:
    print(f, os.path.getsize(f) // 1024, "KB")

out = pathlib.Path(DRIVE) / "runs" / RUN
if (out / "model.pt").exists() and not OVERWRITE:
    raise SystemExit(f"{out} ya tiene model.pt; pon OVERWRITE = True para reentrenar")
out.mkdir(parents=True, exist_ok=True)
pathlib.Path("runs").mkdir(exist_ok=True)
link = pathlib.Path("runs") / RUN
if link.is_symlink() or link.exists():
    link.unlink()
link.symlink_to(out)                  # scripts.train guarda en runs/<RUN>: así cae directo en Drive
print("modelo ->", out)

In [ ]:
extra = " ".join(f"--extra {e}" for e in EXTRA)
# Misma receta que runs/_scripts/pipeline-run.sh (Temis 2.4.0)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u -m scripts.train --name {RUN} --base BSC-LT/MrBERT-es --lr 4e-5 --head-lr 5e-5 --epochs 4 --bs 8 --accum 2 --long-bs 4 --max-len 640 --long-max-len 1536 --ckpt-tokens {CKPT_TOKENS} {extra} 2>&1 | tee -a {DRIVE}/runs/{RUN}/train.log


## Al terminar

En `Mi unidad/temis/runs/<corrida>/` quedan `model.pt` (unos 600 MB), `meta.json` y `train.log`. Copia esa carpeta a `runs/<corrida>/` en el computador de casa; desde allí se exporta, se calibra y se evalúa:

```sh
python -m scripts.export_onnx --name <corrida>
python -m scripts.calibrate --name <corrida>
TEMIS_BACKEND=torch python -m scripts.evaluate --model runs/<corrida>/onnx   # y el resto de evaluaciones
```